In [ ]:
!pip install ultralytics albumentations cvzone

In [ ]:
import torch
from ultralytics import YOLO
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import os

In [ ]:
import numpy as np
from ultralytics import YOLO
import cv2
import cvzone
import math
# from sort_tracker import Sort


cap = cv2.VideoCapture('/kaggle/input/iiuc-idcard-data/test/VID20241006095229.mp4')


if not cap.isOpened():
    print("Error: Could not open video.")
    exit()
    

model = YOLO("/kaggle/working/runs/detect/train3/weights/best.pt")

classNames = ['not_wearing_idcard','wearing_idcard']

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# fps = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('iiuc-idcard-38map.mp4', fourcc, 23, (width,height))

# tracker = Sort(max_age=20,min_hits=3,iou_threshold=0.3)

while True:
    success, img = cap.read()
    if not success:
        print("Error: Could not read frame.")
        break
#     img = cv2.resize(img,(1280,720))

    
    results = model(img, stream=True)
    for r in results:
        #bounding boxex
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            w, h = x2 - x1, y2 - y1 
            

            

            conf = math.ceil(box.conf[0] * 100) / 100
            class_name = int(box.cls[0])
            # print(class_name)
            if classNames[class_name] =='not_wearing_idcard':
                cvzone.cornerRect(img, (x1, y1, w, h),l=5,rt=2,colorR=(0,0,255),colorC=(0,0,150))
                cvzone.putTextRect(img, f"{classNames[class_name]}{conf}", (max(0, x1), max(35, y1)),
                                 scale=1,thickness=1,offset=4,colorR=(0,0,255))
            else:
                cvzone.cornerRect(img, (x1, y1, w, h),l=5,rt=2,colorC=(0,255,0),colorR=(0,150,0))
                cvzone.putTextRect(img, f"{classNames[class_name]}{conf}", (max(0, x1), max(35, y1)),
                                 scale=1,thickness=1,offset=4,colorR=(0,255,0),)
    #             currentArray = np.array([x1,y1,x2,y2,conf])
    #             detections = np.vstack((detections,currentArray))

    # resultsTracker = tracker.update(detections)

#     cv2.imshow("webcam", img)
#     cv2.waitKey(0)
    out.write(img)
cap.release()
out.release()

In [ ]:


# Custom dataset class for loading images and applying Albumentations transformations
class YOLODataset(Dataset):
    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transform = transform
        self.image_paths = [os.path.join(images_dir, x) for x in os.listdir(images_dir)]
        self.label_paths = [os.path.join(labels_dir, x.replace(".jpg", ".txt")) for x in os.listdir(images_dir)]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label_path = self.label_paths[idx]

        # Load image
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Load labels
        with open(label_path, 'r') as f:
            bboxes = []
            class_labels = []
            for line in f.readlines():
                class_id, x_center, y_center, width, height = map(float, line.strip().split())
                bboxes.append([x_center, y_center, width, height])
                class_labels.append(int(class_id))

        # Apply transformations
        if self.transform:
            transformed = self.transform(image=image, bboxes=bboxes, class_labels=class_labels)
            image = transformed['image']
            bboxes = transformed['bboxes']
            class_labels = transformed['class_labels']

        # Return image and labels
        return image, bboxes, class_labels

# Define the Albumentations transformation pipeline
transform = A.Compose([
    A.RandomCrop(width=1024, height=1024),
    A.Rotate(limit=10, p=0.5),
    A.RandomScale(scale_limit=0.2, p=0.5),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Blur(p=0.1),
    A.CLAHE(p=0.1),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2, p=0.5),
    ToTensorV2()
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))


train_dataset = YOLODataset(
    images_dir='/kaggle/input/iiuc-idcard-data/train/images',
    labels_dir='/kaggle/input/iiuc-idcard-data/train/labels',
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

model = YOLO("/kaggle/working/runs/detect/train2/weights/best.pt")

model.train(data="/kaggle/input/iiuc-idcard-data/data.yaml", epochs=300,batch=32, imgsz=640, lr0=0.000005,weight_decay=0.0005)

model.save("/kaggle/working/yolo_model_11m_v2.pt")

print("Training completed and model saved successfully.")
